# Manuscript simulation: SMI, SSST-CSD and MSMT-CSD on a 60 degree crossing

The manuscript figure source, as a notebook. It is the same experiment as
`smi_manuscript_60deg.m` — same ground truth, same protocol, same seeds, same
`CHECK` discipline — with **the CSD and MSMT-CSD arms wired in for real**, and
cut down to the **healthy white matter kernel only**.

**Three arms, one peak finder.**

| arm | what it is |
|---|---|
| **SMI** | `SMI.fit` with `fODF_regularization.flag_nonneg = 1`, `CS_phase = 0` |
| **SSST-CSD** | `dwi2fod csd` on the top shell — MRtrix3 3.0.4, the binary |
| **MSMT-CSD** | `dwi2fod msmt_csd` on all shells, 3 tissues — the binary |

Nothing here reimplements MRtrix. `dwi2fod`, `dwiextract`, `mrinfo` and
`sh2peaks` are called as subprocesses; the only MRtrix behaviour implemented
locally is reading and writing its image format (`mrtrix_io.m`), which is
verified against `mrinfo` on every run.

**The responses are idealized, not estimated.** Every arm is given the *exact*
zonal response of the kernel that generated the data,
`r_l(b) = K_l(b) sqrt((2l+1) 4 pi)`, evaluated at the b values MRtrix itself
reports for each shell. That is deliberate: it removes response estimation as a
confound, so a difference between arms is the deconvolution and nothing else.
A response estimated from data is 15–40% blunter than this
(`Reports/REPORT_SMI_deconvolution_MonteCarlo.md`, section 6.3), and swapping
one in is a one-line change in Step 6b.

**No edema kernel here.** The two-kernel comparison lives in
`smi_manuscript_60deg.m`. This notebook is the healthy baseline the edema arm
will be measured against once a synthetic edema response is supplied.

### What you should end up believing

Not "the code ran". Every step ends in one or more **CHECK** lines that compare
its output against something computed a *different* way, and print `ok` or
`** FAILED **` next to the number. Reading only the CHECK lines is a complete
audit.

```
 Step 1   the acquisition protocol          -> a real HCP 3-shell scheme
 Step 2   the ground truth fibre geometry   -> a 60 degree crossing
 Step 3   the kernel, and it as a response  -> K_l(b), zonal harmonics
 Step 4   forward convolution               -> noise-free signal
 Step 5   Rician noise, one block per SNR   -> the measured data
 Step 6   SMI.fit at each Lmax and each SNR -> the SMI arm
 Step 6b  dwi2fod csd / msmt_csd            -> the two MRtrix arms
 Step 7   peaks, angular error, spurious    -> where does it stop working?
 Step 8   export + sh2peaks cross-check     -> an independent peak finder
 Figures  1-4
 Step 9   how to scale this up
```

### Runtime

`SMI.fit` is the whole cost: it trains a polynomial regression once per call
regardless of voxel count, so each `(Lmax, SNR)` pair costs a couple of minutes
whatever `NREP` is. The two MRtrix arms together are seconds. The notebook ships
with **`SMOKE_TEST = true`**, which runs one Lmax and three SNRs in a few
minutes; set it to `false` for the manuscript configuration, which is
`numel(SNR_LIST) * numel(LMAX_LIST)` = 21 fits and runs in hours.

In [ ]:
% ============================ Configuration ============================
% Every knob in this simulation is here. Nothing below this cell needs editing
% to retune the experiment.
%
% The three things this notebook still takes from mc_config.m are stateless
% UTILITIES, not settings -- pick_grid, rotate_about and load_protocol_file.
% They are shared rather than copied so the fibre-axis convention and the
% protocol reader cannot drift away from gen_montecarlo.m.

warning('off','all');
more off;

% Locate deconv_comparison/ by walking up from wherever the kernel started.
% mfilename('fullpath') is empty in a Jupyter kernel, so the .m file's trick of
% asking for its own path does not work here.
d = pwd; pkgdir = '';
for k = 1:6
    if exist(fullfile(d,'deconv_comparison','oct_path.m'), 'file')
        pkgdir = fullfile(d,'deconv_comparison'); break
    elseif exist(fullfile(d,'oct_path.m'), 'file')
        pkgdir = d; break
    end
    dn = fileparts(d); if strcmp(dn,d), break; end; d = dn;
end
if isempty(pkgdir)
    error('cannot find deconv_comparison/oct_path.m from %s', pwd);
end
run(fullfile(pkgdir,'oct_path.m'));
fprintf('package     : %s\n', pkgdir);

MC = mc_config();                        % utilities only, see above
H  = fODF_modulation_helpers();
RH = SMI_response_helpers();
MR = mrtrix_io();
VERDICT = {'** FAILED **', 'ok'};        % VERDICT{1+condition}

% -------------------------------------------------------------- the tissue
% SMI's compartment vector is [f Da Depar Deperp fw]:
%   f        intra-axonal (stick) fraction
%   Da       intra-axonal diffusivity along the stick        um^2/ms
%   Depar    extra-axonal diffusivity parallel to the fibre  um^2/ms
%   Deperp   extra-axonal diffusivity perpendicular          um^2/ms
%   fw       free water fraction, D fixed at D_FW below
% The extra-axonal fraction is whatever is left: 1 - f - fw.
%
% ONE kernel here: healthy white matter, the kernel the published Monte Carlo
% used. The edema kernel is not simulated in this notebook.
PRESET = 'healthy';
K_WM   = [0.60 2.0 2.0 0.50 0.02];
D_FW   = 3;         % free water diffusivity, um^2/ms
D_GM   = 0.8;       % grey matter diffusivity, um^2/ms -- ONLY used to build the
                    % idealized isotropic response MSMT-CSD needs as its second
                    % tissue. No simulated voxel contains grey matter.
KAPPA  = 16;        % Watson concentration of each fibre population. Finite, not
                    % a delta: a response estimated from real white matter has
                    % already absorbed fibre dispersion, and a delta truth would
                    % create a mismatch that does not exist in practice.

% ------------------------------------------------------------- the geometry
ANGLES = 60;        % crossing angles in degrees. Measured on the noise-free
                    % truth, 60 separates at every Lmax and every KAPPA tested,
                    % where 30 never separates at KAPPA = 16 and 45 only from
                    % Lmax 6 up. That is what makes a difference between methods
                    % attributable to the method rather than to the band limit.
AXIS1  = [0.30 -0.50 0.81];       % first fibre axis, off every coordinate plane
AXIS1  = AXIS1/norm(AXIS1);

% ------------------------------------------------------------ the experiment
SMOKE_TEST = true;                % <-- THE KNOB. false = manuscript settings.

if SMOKE_TEST
    NREP      = 27;               % realisations per condition PER SNR
    SNR_LIST  = [10 30 Inf];
    LMAX_LIST = 6;
    GLYPH_N   = 61;               % glyph mesh resolution
else
    NREP      = 1000;
    SNR_LIST  = [5 10 20 30 50 100 Inf];
    LMAX_LIST = [4 6 8];
    GLYPH_N   = 121;
end
% NREP * numel(ANGLES) and NREP * numel(ANGLES) * numel(SNR_LIST) must each
% factor into three integers > 1: SMI.vectorize takes a different branch if any
% spatial dimension is a singleton. pick_grid errors loudly if they do not.

LMAX_GT   = 8;                    % ground truth angular order. A CEILING, not a
                                  % choice: SMI's kernel invariants K_l are
                                  % undefined above l = 8.
CS_PHASE  = 0;                    % 0 == MRtrix's SH basis exactly. At SMI's
                                  % default of 1 the two differ by (-1)^m, a 180
                                  % degree rotation about z of every fODF. This
                                  % is what lets an SMI fODF and an MRtrix fODF
                                  % be scored by the same lines of code.
PROTOCOL  = 'hcp_real_3shell.txt';
B0_SNAP   = 0.05;                 % ms/um^2. Any shell below this is treated as
                                  % exactly b = 0. The acquired .bval carries
                                  % EFFECTIVE b, so its "b = 0" volumes are
                                  % b = 5 s/mm^2. Snapping restores the exact
                                  % identity S(0)/S0 = 1. The diffusion-weighted
                                  % shells keep their acquired jitter.
NDIR_Q    = 3000;                 % quadrature directions for projecting a
                                  % sampled fODF onto plm
SEED      = 31415;                % RNG seed

% ---------------------------------------------- the constrained deconvolution
% Passed straight through to options.fODF_regularization. flag_nonneg = 1 is the
% arm being studied; it is OFF in the shipped toolbox defaults.
REG = struct('flag_nonneg', 1, 'lambda_tikhonov', 0.3);

% ------------------------------------------------------------ the MRtrix arms
RUN_MRTRIX   = true;    % false skips Step 6b and scores the SMI arm alone
RUN_SH2PEAKS = true;    % Step 8's independent cross-check against sh2peaks
MDIR = fullfile(pkgdir, 'mrtrix');      % gitignored scratch for MRtrix images
TAG  = ['nb_' PRESET];

% ------------------------------------------------------------------ assembled
C = struct();
C.K_WM = K_WM;  C.PRESET = PRESET;  C.D_FW = D_FW;  C.KAPPA = KAPPA;
C.ANGLES = ANGLES;  C.AXIS1 = AXIS1;  C.LMAX_GT = LMAX_GT;
C.CS_PHASE = CS_PHASE;  C.PROTOCOL = PROTOCOL;  C.NDIR_Q = NDIR_Q;
C.SEED_MC = SEED;
C.pick_grid     = MC.pick_grid;                  % shared utilities
C.rotate_about  = MC.rotate_about;
C.load_protocol = @() MC.load_protocol_file(PROTOCOL);

AX = cell(1, numel(ANGLES));
for ic = 1:numel(ANGLES)
    if ANGLES(ic) == 0
        AX{ic} = {AXIS1};
    else
        AX{ic} = {AXIS1, C.rotate_about(AXIS1, ANGLES(ic))};
    end
end
C.condition_axes = @(ic) AX{ic};

NSNR       = numel(SNR_LIST);
SIGMA_LIST = 1./SNR_LIST;                     % Inf -> 0
SNR_LABEL  = cell(1, NSNR);
for is = 1:NSNR
    if isinf(SNR_LIST(is)), SNR_LABEL{is} = 'inf';
    else, SNR_LABEL{is} = sprintf('%g', SNR_LIST(is)); end
end
[~, SNR_ORD] = sort(SNR_LIST);   % ascending, Inf last: the order every figure
                                 % and every table uses, whatever order
                                 % SNR_LIST was typed in

fprintf('\n=== SMI / SSST-CSD / MSMT-CSD manuscript simulation ===\n');
if SMOKE_TEST
    fprintf('*** SMOKE_TEST = true: reduced sweep, indicative numbers only ***\n');
end
fprintf('kernel      : %-8s [f Da Depar Deperp fw] = %s, extra-axonal = %.2f\n', ...
        PRESET, mat2str(K_WM), 1 - K_WM(1) - K_WM(5));
fprintf('condition   : %g deg crossing only,  kappa = %g\n', ANGLES, KAPPA);
fprintf('protocol    : %s,  CS_phase %d,  truth at Lmax %d\n', PROTOCOL, CS_PHASE, LMAX_GT);
fprintf('sweep       : SNR %s x Lmax %s = %d SMI.fit calls of %d voxels each\n', ...
        mat2str(SNR_LIST), mat2str(LMAX_LIST), NSNR*numel(LMAX_LIST), ...
        numel(ANGLES)*NREP);
fprintf('MRtrix arms : %s\n\n', VERDICT{1+RUN_MRTRIX});

## Which Lmax, and why the ground truth is stuck at 8

Every result below depends on this, and the numbers mean different things at
different orders.

**The fit runs at each entry of `LMAX_LIST`.** Even spherical harmonics up to
Lmax number `(Lmax/2+1)(Lmax+1)` = 15, 28 and 45 coefficients at Lmax 4, 6, 8.
The protocol has 90 directions per shell, so all three are comfortably
determined by the data; the limit is the model and the noise, not the sampling.

**The ground truth is built at Lmax 8, and that is a ceiling, not a choice.**
SMI's kernel rotational invariants `K_l(b)` are only defined up to `l = 8`
(`SMI.RotInv_Kell_wFW_b_beta_TE_numerical`; ask for `l = 10` and it errors). So
the truth cannot be made sharper than the highest order being fitted.

- At **Lmax 4 and 6** the truth carries angular detail the fit cannot represent,
  which is the honest arrangement.
- At **Lmax 8** the truth is exactly representable by the fit. The "ceiling"
  reported in Step 7 is then no longer an independent bound, and any Lmax 8
  result should be read as slightly flattering.

The published comparison in `Reports/` ran every arm at **Lmax 6**, so 6 is the
row to compare against the report, and it is the one `SMOKE_TEST` keeps.

## Step 1 — the acquisition protocol

A **real HCP 3-shell scheme**, supplied as FSL `.bval` / `.bvec` and tracked as
text at `protocol/hcp_real_3shell.txt`. 288 volumes: 18 at b ≈ 0 plus 90 each at
nominal b = 1, 2, 3 ms/µm². b is carried in ms/µm² throughout; the tracked table
is the supplied values divided by 1000.

Two properties of real acquisitions show up immediately, and neither is tidied
away, because both are things code that only ever saw synthetic protocols would
get wrong:

- **The b = 0 volumes are not b = 0 as acquired** — they are b = 5 s/mm², because
  a `.bval` records *effective* b. `B0_SNAP` sets them to exactly 0.
- **The b values jitter within each shell** — 18 distinct values across the
  scheme. The forward model uses the exact per-volume b; SMI bins them into
  shells for the kernel fit, and this step checks it bins them the way a human
  would. **Step 6b checks that MRtrix bins them the same way**, which is not
  automatic and is what makes the response rows line up with the right shells.

Expect a loud `WARNING` about gradient directions: the supplied `.bvec` is unit
only to `1.1e-6`, being written at seven significant figures. It is deliberately
noisy — left uncorrected it degrades the zonal-response identity at Lmax 8 from
`1e-15` to `5e-7`, and that failure is invisible unless something checks for it.

In [ ]:
[bvals, bvecs] = C.load_protocol();
Ndwi = numel(bvals);

n_snap = sum(bvals > 0 & bvals < B0_SNAP);
if n_snap > 0
    fprintf(['   note: %d volumes with 0 < b < %g snapped to exactly b = 0\n' ...
             '         (acquired as b = %g ms/um^2 = %.0f s/mm^2, effective b\n' ...
             '          from the imaging gradients)\n'], ...
            n_snap, B0_SNAP, max(bvals(bvals < B0_SNAP)), ...
            1000*max(bvals(bvals < B0_SNAP)));
    bvals(bvals < B0_SNAP) = 0;
end

fprintf('Step 1: %d volumes, %d distinct b values\n', Ndwi, numel(unique(bvals)));

% Let SMI group the shells, and report what it decided. This is the function the
% fit itself uses, and its shell assignment is used for every per-shell number
% below.
[tbl, ~, shell_id] = SMI.Group_dwi_in_shells_b_beta_TE(bvals, [], [], []);
b_shell = tbl(1,:);
n_shell = tbl(3,:);
for i = 1:numel(b_shell)
    raw = bvals(shell_id == i);
    fprintf('   shell %d: b = %.3f ms/um^2, %3d directions, raw b in [%.3f, %.3f]\n', ...
            i, b_shell(i), n_shell(i), min(raw), max(raw));
end
dw = shell_id(:)' > 1;                        % everything above the b~0 shell

% CHECK 1. Every direction must be a unit vector. This passes trivially because
% the loader normalised them -- the number that matters is the one in the
% warning above, which is the error as the .bvec was supplied.
e_nrm = max(abs(sqrt(sum(bvecs.^2, 2)) - 1));
fprintf('   CHECK all directions are unit    max| |g|-1 | = %.2e   %s\n', ...
        e_nrm, VERDICT{1+(e_nrm < 1e-12)});

% CHECK 2. SMI must recover the four shells a human sees, from 18 distinct b
% values. If it split a shell, every rotational invariant downstream would be
% estimated from a fraction of the directions.
ok_shell = (numel(b_shell) == 4) && all(n_shell(:)' == [18 90 90 90]);
fprintf('   CHECK SMI bins %d b values into 4 shells [18 90 90 90]   %s\n', ...
        numel(unique(bvals)), VERDICT{1+ok_shell});

% CHECK 3. Within each diffusion-weighted shell the directions should cover the
% sphere evenly. A clumped scheme would make the l >= 4 coefficients unstable in
% a way that looks like a method failure.
fprintf('   nearest-neighbour direction spacing per shell:\n');
for i = 2:numel(b_shell)
    g = bvecs(shell_id == i, :);
    cc = abs(g*g'); cc(1:size(g,1)+1:end) = 0;
    nn = acosd(min(max(max(cc, [], 2), -1), 1));
    fprintf('     b = %.0f : mean %.2f deg, worst gap %.2f deg\n', ...
            b_shell(i), mean(nn), max(nn));
end
fprintf('\n');

## Step 2 — the ground truth fibre geometry

Two equal fibre populations crossing at 60 degrees. Each population is a
**Watson** distribution with concentration `kappa = 16`, not a delta.

That choice is the one most likely to be questioned, so: a response function
estimated from real white matter has already absorbed fibre dispersion. A delta
ground truth would create a response/truth mismatch that does not exist in
practice, and would flatter whichever method sharpens most. It also matters here
specifically, because the idealized response given to CSD in Step 6b *is* a
delta response — so the dispersion in the truth is the only thing standing
between the arms and a trivially exact answer.

The fODF is sampled on `NDIR_Q` quadrature directions and projected onto
spherical harmonic coefficients `plm` in SMI's normalised convention `p_00 = 1`,
in which **the fODF integrates to 1 in every voxel**.

In [ ]:
dq    = H.dirs(C.NDIR_Q);
NCOND = numel(C.ANGLES);
COLHDR = cell(1, NCOND);
for ic = 1:NCOND
    if C.ANGLES(ic) == 0, COLHDR{ic} = 'single';
    else, COLHDR{ic} = sprintf('%d deg', C.ANGLES(ic)); end
end
L_gt = repelem(0:2:LMAX_GT, 2*(0:2:LMAX_GT)+1)';

fodf_gt = zeros(size(dq,1), NCOND);
plm_gt  = zeros(NCOND, numel(L_gt)-1);
sh_gt   = zeros(NCOND, numel(L_gt));
axes_gt = cell(1, NCOND);
sep_deg = zeros(1, NCOND);

fprintf('Step 2: %d condition(s)\n', NCOND);
for ic = 1:NCOND
    axes_gt{ic} = C.condition_axes(ic);
    f = zeros(size(dq,1), 1);
    for k = 1:numel(axes_gt{ic})
        f = f + H.watson(dq, axes_gt{ic}{k}, C.KAPPA);
    end
    fodf_gt(:,ic) = f;
    p             = H.mixture_plm(f, dq, LMAX_GT, C.CS_PHASE);
    plm_gt(ic,:)  = p(:)';
    sh_gt(ic,:)   = ([1; p(:)] .* sqrt((2*L_gt+1)/(4*pi)))';
    if numel(axes_gt{ic}) == 2
        sep_deg(ic) = acosd(abs(axes_gt{ic}{1} * axes_gt{ic}{2}'));
    end
    fprintf('   condition %d: %2d deg, %d population(s), measured separation %.6f deg\n', ...
            ic, C.ANGLES(ic), numel(axes_gt{ic}), sep_deg(ic));
end

% CHECK 1. The angle between the two axes must be the angle we asked for. A
% wrong rotation axis would otherwise look plausible all the way to the end.
e_sep = max(abs(sep_deg - C.ANGLES));
fprintf('   CHECK crossing angles       max err  = %.2e deg   %s\n', ...
        e_sep, VERDICT{1+(e_sep < 1e-9)});

% CHECK 2. Reconstruct each fODF from its plm and integrate it over the sphere.
% In the p_00 = 1 convention the integral is 1 by construction.
Yq     = SMI.get_even_SH(dq, LMAX_GT, C.CS_PHASE);
amp    = sh_gt * Yq';
mass   = mean(amp, 2) * 4*pi;                 % equal-area quadrature
e_mass = max(abs(mass - 1));
fprintf('   CHECK fODF integrates to 1  max|int-1| = %.2e   %s\n', ...
        e_mass, VERDICT{1+(e_mass < 1e-3)});

% CHECK 3. The fODF as *sampled* must be non-negative: it is a sum of Watson
% distributions, which are positive by construction. (These are unnormalised
% Watson amplitudes, so the floor is 1, not 0.)
mn_q = min(fodf_gt(:));
fprintf('   CHECK sampled fODF >= 0     min = %+.4f            %s\n', ...
        mn_q, VERDICT{1+(mn_q >= 0)});

% NOT a check, but the single most surprising number in this notebook:
% the band-limited ground truth is itself NEGATIVE in places. Truncating a
% Watson mixture rings, exactly as truncating any Fourier series does, and the
% rings go below zero. So the "truth" the non-negativity constraint in Step 6 is
% asked to approach does not satisfy non-negativity either. This is why the
% constraint is a regularizer and not a statement of fact -- and it applies just
% as much to the CSD arms, whose constraint is the same idea.
fprintf('   band-limited truth at Lmax %d -- peak, minimum, %% of sphere below 0:\n', LMAX_GT);
for ic = 1:NCOND
    fprintf('     %2d deg : peak %+.4f, min %+.4f, negative over %4.1f%% of directions\n', ...
            C.ANGLES(ic), max(amp(ic,:)), min(amp(ic,:)), 100*mean(amp(ic,:) < 0));
end
fprintf('   isotropic floor 1/(4*pi) = %.4f  (MRtrix iFOD2 default cutoff is 0.05)\n\n', ...
        1/(4*pi));

## Step 3 — the kernel, and the same kernel as a response function

SMI has no response function. It has a **kernel**: the Standard Model
compartment description `[f Da Depar Deperp fw]`, from which the rotational
invariants `K_l(b)` follow analytically. CSD has the opposite arrangement, a
non-parametric response estimated once and stored as zonal harmonics.

**They describe the same object.** For a single fibre along z,

```
R(theta) = sum_l K_l(b) (2l+1) P_l(cos theta) = sum_l r_l Y_l0(theta)
r_l      = K_l(b) * sqrt((2l+1)*4*pi)
```

and `r_l` is exactly one row of an MRtrix response `.txt` file. That identity is
what lets an SMI kernel be handed to MRtrix at all, so it is checked here rather
than assumed — and Step 6b checks it a second time, on the file MRtrix actually
reads, after a round trip through disk.

In [ ]:
K = C.K_WM;
fprintf('Step 3: kernel [f Da Depar Deperp fw] = %s\n', mat2str(K));
fprintf('   zonal response r_l at the nominal shells -- this IS an MRtrix response file:\n');
fprintf('        b   ');
for l = 0:2:LMAX_GT, fprintf('%10s', sprintf('r_%d', l)); end
fprintf('\n');
b_nom = [0 1 2 3];
r_nom = RH.zh(K, b_nom, LMAX_GT, C.D_FW);
for i = 1:numel(b_nom)
    fprintf('     %4g   ', b_nom(i)); fprintf('%10.4f', r_nom(i,:)); fprintf('\n');
end
fprintf('   normalised to l = 0, which is how a response is usually quoted:\n');
for i = 1:numel(b_nom)
    fprintf('     %4g   ', b_nom(i)); fprintf('%10.4f', r_nom(i,:)/r_nom(i,1)); fprintf('\n');
end
fprintf('   r_l falls fast with l: that decay is what makes deconvolution ill-conditioned,\n');
fprintf('   and it is why Lmax 8 is harder than Lmax 4 rather than simply better.\n');

% CHECK. Build the signal of a delta fODF along z two ways and compare.
% Route A, the zonal profile above. Route B, SMI's own forward model fed the
% plm of a delta. In the p_00 = 1 convention a delta along z is exact and needs
% no approximation: every p_l0 = 1 and every other p_lm = 0.
Mf = [];
for l = 2:2:LMAX_GT, Mf = [Mf, -l:l]; end
plm_delta = double(Mf(:)' == 0);

zax   = [0 0 1];
bsub  = bvals(dw);
gsub  = bvecs(dw,:);
S_fwd = H.signal(plm_delta, [K 1 1], bsub, ones(1,sum(dw)), zeros(1,sum(dw)), ...
                 gsub, LMAX_GT, C.CS_PHASE, C.D_FW);
theta = acos(max(min(gsub * zax', 1), -1));
r_each = RH.zh(K, bsub, LMAX_GT, C.D_FW);     % the exact per-volume b, jitter included
S_zon  = zeros(sum(dw), 1);
for i = 1:sum(dw)
    S_zon(i) = RH.profile(r_each(i,:), theta(i));
end
e_zon = max(abs(S_zon(:) - S_fwd(:)));
fprintf('   CHECK zonal response == SMI forward model  max|err| = %.2e   %s\n\n', ...
        e_zon, VERDICT{1+(e_zon < 1e-9)});

## Step 4 — forward convolution to a noise-free signal

The signal is the fODF convolved with the kernel. In spherical harmonics
convolution is a product:

```
S(u)/S0 = sum_lm K_l(b) p_lm Y_lm(u) sqrt((2l+1)*4*pi)
```

This is exactly the expression `SMI.get_plm_from_S_and_kernel` inverts. Using it
here is less circular than it looks — the test is whether the *inverse* recovers
the input, which is Steps 6 and 7 — but to make sure the harmonic machinery
itself is right, the same signal is also computed with **no spherical harmonics
anywhere**, by direct numerical convolution over the quadrature grid:

```
S(u) = sum_q w_q * fODF(n_q) * K(u.n_q),   with sum_q w_q = 1
```

In [ ]:
S_clean = zeros(NCOND, Ndwi);
for ic = 1:NCOND
    s = H.signal(plm_gt(ic,:), [K 1 1], bvals, ones(1,Ndwi), zeros(1,Ndwi), ...
                 bvecs, LMAX_GT, C.CS_PHASE, C.D_FW);
    S_clean(ic,:) = s(:)';
end

f_ = K(1); Da_ = K(2); Dep_ = K(3); Dpp_ = K(4); fw_ = K(5);
cosang  = bvecs * dq';
Kmat    = f_         * exp(-bvals' .* (Da_*cosang.^2)) + ...
          (1-f_-fw_) * exp(-bvals' .* (Dpp_ + (Dep_-Dpp_)*cosang.^2)) + ...
          fw_        * exp(-bvals' * C.D_FW);
S_exact = zeros(NCOND, Ndwi);
for ic = 1:NCOND
    w = fodf_gt(:,ic) / sum(fodf_gt(:,ic));
    S_exact(ic,:) = (Kmat * w)';
end

fprintf('Step 4: noise-free signal\n');
fprintf('   mean signal per shell (shells as SMI assigned them in Step 1):\n');
fprintf('        b   ');  fprintf('%10s', COLHDR{:}); fprintf('\n');
for i = 1:numel(b_shell)
    fprintf('     %4.2f   ', b_shell(i));
    fprintf('%10.4f', mean(S_clean(:, shell_id == i), 2)); fprintf('\n');
end

% CHECK 1. The b~0 signal must be essentially 1, because sigma is set to 1/SNR
% below and that only means the requested SNR if S0 = 1.
S_b0 = mean(S_clean(:, ~dw), 2);
e_s0 = max(abs(S_b0 - 1));
if B0_SNAP > 0
    fprintf('   CHECK S(b=0) == 1 exactly        max|S-1| = %.2e   %s\n', ...
            e_s0, VERDICT{1+(e_s0 < 1e-12)});
else
    fprintf('   CHECK S(b~0) is within 1%% of 1   max|S-1| = %.2e   %s\n', ...
            e_s0, VERDICT{1+(e_s0 < 0.01)});
end

% CHECK 2. Harmonics against direct convolution. These do NOT agree to machine
% precision, and should not: the harmonic route is band limited at LMAX_GT and
% the direct sum is not. The residual IS the band-limiting error of the ground
% truth, and Step 5 puts it next to the noise.
e_sh = max(abs(S_clean(:) - S_exact(:)));
fprintf('   CHECK harmonics vs direct convolution  max|err| = %.2e\n', e_sh);
fprintf('         (band limiting, not an error -- it falls as LMAX_GT rises)\n\n');

## Step 5 — Rician noise, one block of voxels per SNR

Complex Gaussian noise is added to a real signal and the magnitude taken, which
is exactly Rician:

```
S_noisy = sqrt( (S + sigma*n1)^2 + (sigma*n2)^2 ),   n1, n2 ~ N(0,1)
```

`sigma = 1/SNR`, with SNR defined against the b = 0 signal.

**The sweep is laid out as one contiguous block of voxels per SNR**, each block
holding all `NCOND` conditions at `NREP` realisations. Step 6 hands one block at
a time to `SMI.fit`, so every fit sees exactly one noise level — which is what
`gen_montecarlo.m` does. **The MRtrix arms in Step 6b see the whole sweep in one
image**, because `dwi2fod` is per-voxel and has no equivalent of SMI's
noise-level-dependent regression training. That asymmetry is real and worth
knowing: it is a structural advantage to the CSD arms at the noisy end, and a
structural disadvantage nowhere.

**Each SNR is seeded separately**, from `SEED` offset by the SNR index, so each
noise level is reproducible on its own and adding or removing an entry from
`SNR_LIST` does not silently change the realisations of every other entry. The
cost is that the blocks do not share common random numbers, so a difference
between two SNRs carries the Monte Carlo error of both.

In [ ]:
NVOX_SNR = NCOND*NREP;                  % voxels in one SNR block: what a fit sees
NVOX     = NVOX_SNR*NSNR;               % voxels in the whole sweep
GRID_SNR = C.pick_grid(NVOX_SNR);       % the 3D grid one SMI.fit call is given
GRID_ALL = C.pick_grid(NVOX);           % the grid the MRtrix arms are given
cond_id  = repmat(repelem((1:NCOND)', NREP, 1), NSNR, 1);
snr_id   = repelem((1:NSNR)', NVOX_SNR, 1);
S_rep    = S_clean(cond_id, :);
S_noisy  = zeros(NVOX, Ndwi);

for is = 1:NSNR
    rows = find(snr_id == is);
    sg   = SIGMA_LIST(is);
    rand('seed', C.SEED_MC + is); randn('seed', C.SEED_MC + is);
    Sb   = S_rep(rows, :);
    % At sigma = 0 this is sqrt(S^2) = S, i.e. the noise-free signal exactly.
    % The randn draws still happen so the stream is the same shape at every SNR.
    S_noisy(rows,:) = sqrt((Sb + sg*randn(size(Sb))).^2 + ...
                           (     sg*randn(size(Sb))).^2);
end

fprintf('Step 5: %d condition(s) x %d reps x %d SNR = %d voxels\n', ...
        NCOND, NREP, NSNR, NVOX);
fprintf('   each SMI.fit sees one SNR block: %d voxels, grid %s\n', ...
        NVOX_SNR, mat2str(GRID_SNR));
fprintf('   the MRtrix arms see the whole sweep at once: %d voxels, grid %s\n', ...
        NVOX, mat2str(GRID_ALL));

% CHECK 1. Recover sigma from the simulated data at every SNR, rather than
% trusting the value typed in. At SNR = inf it is the statement that the "noisy"
% signal is bit identical to the noise-free one.
for is = 1:NSNR
    rows  = find(snr_id == is);
    rblk  = S_noisy(rows,:) - S_rep(rows,:);
    s_hat = std(rblk(:));
    sg    = SIGMA_LIST(is);
    if sg == 0
        ok_is = (max(abs(rblk(:))) == 0);
        fprintf('   CHECK SNR %-4s  noise free, residual is exactly zero        %s\n', ...
                SNR_LABEL{is}, VERDICT{1+ok_is});
    else
        rel   = abs(s_hat - sg)/sg;
        fprintf('   CHECK SNR %-4s  recovered sigma %.5f vs %.5f, %4.1f%% off  %s\n', ...
                SNR_LABEL{is}, s_hat, sg, 100*rel, VERDICT{1+(rel < 0.10)});
    end
end

% Where the band limit sits relative to the noise.
sg_min = min(SIGMA_LIST(SIGMA_LIST > 0));
if isempty(sg_min)
    fprintf('   every SNR in the sweep is noise free, so the truth''s band-limiting\n');
    fprintf('   error %.2e is the only error anywhere below\n', e_sh);
else
    fprintf('   smallest non-zero sigma %.4f is %.0fx the band-limiting error %.2e,\n', ...
            sg_min, sg_min/e_sh, e_sh);
    fprintf('   so noise dominates the band limit at every finite SNR in the sweep\n');
end

% CHECK 2. A magnitude is strictly positive, which is why SMI is told
% NoiseBias = 'Rician' in Step 6.
mn = min(S_noisy(:));
fprintf('   CHECK signal strictly positive   min = %.4f            %s\n', ...
        mn, VERDICT{1+(mn > 0)});

% The Rician floor, per SNR: the bias the magnitude operation puts into the
% highest shell. Note that NOTHING in the CSD arms corrects for this -- dwi2fod
% has no noise model -- while SMI is told about it explicitly. That is a real
% difference between the arms at the bottom of the sweep, not a setup error.
hib = (shell_id(:)' == numel(b_shell));
fprintf('   Rician floor at b = %.0f, mean over the shell:\n', b_shell(end));
fprintf('        SNR     noisy   noise free      upward bias\n');
for is = 1:NSNR
    rows = find(snr_id == is);
    fprintf('     %6s  %8.4f     %8.4f     %+8.4f\n', SNR_LABEL{is}, ...
            mean(mean(S_noisy(rows,hib))), mean(mean(S_rep(rows,hib))), ...
            mean(mean(S_noisy(rows,hib))) - mean(mean(S_rep(rows,hib))));
end

% NOT a check, but the thing about a sweep that is easiest to get wrong:
% SMI does not fit the kernel at the sigma you pass in. It normalises sigma by
% the measured b = 0 signal, bins the result into Nlevels equal bins spanning
% sigma_norm_limits, and trains one polynomial regression per occupied bin,
% evaluated at the bin CENTRE (SMI.m:2222-2302). Both constants are hard coded
% in SMI.m, so they are restated here rather than read back.
SMI_SIGMA_LIMITS = [0 0.2];             % SMI.m:562
SMI_NLEVELS      = 10;                  % SMI.m:388, the default Nlevels
lev_w = (SMI_SIGMA_LIMITS(2)-SMI_SIGMA_LIMITS(1))/SMI_NLEVELS;
fprintf('   how SMI will bin these noise levels (Nlevels %d over sigma_norm %s):\n', ...
        SMI_NLEVELS, mat2str(SMI_SIGMA_LIMITS));
fprintf('        SNR     sigma   bin   trained at\n');
for is = 1:NSNR
    lev = min(max(floor(SIGMA_LIST(is)/lev_w) + 1, 1), SMI_NLEVELS);
    fprintf('     %6s  %8.4f  %4d     %8.4f', SNR_LABEL{is}, SIGMA_LIST(is), ...
            lev, (lev-0.5)*lev_w);
    if SIGMA_LIST(is) > SMI_SIGMA_LIMITS(2)
        fprintf('   <-- ABOVE the trained range, clamped');
    end
    fprintf('\n');
end
fprintf('   These bins are nominal: sigma is normalised by the MEASURED b = 0\n');
fprintf('   signal, so voxels of one SNR can straddle a bin edge.\n\n');

## Step 6 — the SMI arm: `SMI.fit`, at each Lmax and each SNR

The real toolbox at its shipped defaults, with the constrained deconvolution
turned on. Two options are not defaults and both matter:

- **`CS_phase = 0`.** At SMI's default of 1 the spherical harmonic basis differs
  from MRtrix's by `(-1)^m`, which for even l is a 180 degree rotation about z
  of every fODF — 71.5 degrees of peak error, verified against MRtrix's own
  `sh2peaks` in `check_mrtrix_basis.sh`. At 0 the two bases are **identical**,
  which is the whole reason all three arms can go through the same scoring code.
- **`fODF_regularization.flag_nonneg = 1`.** Off by default in the toolbox; this
  is the arm being studied. `lambda_nonneg` is left at its shipped value of 1
  and printed back as confirmation.

In [ ]:
SH_SMI    = cell(1, numel(LMAX_LIST));   % SH_SMI{iL} = [NVOX x ncoef]
KERN_SMI  = cell(1, numel(LMAX_LIST));
CONV_SMI  = cell(1, numel(LMAX_LIST));
NONFIN    = zeros(1, numel(LMAX_LIST));

fprintf('Step 6: %d SMI.fit calls (%d Lmax x %d SNR), %d voxels each\n', ...
        numel(LMAX_LIST)*NSNR, numel(LMAX_LIST), NSNR, NVOX_SNR);
for iL = 1:numel(LMAX_LIST)
    Lf    = LMAX_LIST(iL);
    Lv    = repelem(0:2:Lf, 2*(0:2:Lf)+1)';
    ncoef = numel(Lv);

    sh_all   = zeros(NVOX, ncoef);
    kern_all = [];
    conv_all = false(NVOX, 1);

    for is = 1:NSNR
        rows  = find(snr_id == is);
        dwi_b = reshape(S_noisy(rows,:), [GRID_SNR Ndwi]);

        options = struct();
        options.b     = bvals;
        options.dirs  = bvecs;
        options.sigma = SIGMA_LIST(is)*ones(GRID_SNR);
        options.mask  = true(GRID_SNR);
        options.compartments  = {'IAS','EAS','FW'};
        options.NoiseBias     = 'Rician';
        options.Lmax          = [0 Lf Lf Lf];
        options.CS_phase      = C.CS_PHASE;
        options.D_FW          = C.D_FW;
        options.flag_fit_fODF = 1;
        options.fODF_regularization = REG;

        t0  = tic;
        out = SMI.fit(dwi_b, options);
        el  = toc(t0);

        plm = reshape(out.plm, [NVOX_SNR ncoef-1]);
        sh  = [ones(NVOX_SNR,1) plm] .* repmat(sqrt((2*Lv+1)/(4*pi))', NVOX_SNR, 1);
        NONFIN(iL) = NONFIN(iL) + sum(~isfinite(sh(:)));
        sh(~isfinite(sh)) = 0;
        sh_all(rows,:) = sh;

        kb = reshape(out.kernel, [NVOX_SNR size(out.kernel,4)]);
        if isempty(kern_all), kern_all = zeros(NVOX, size(kb,2)); end
        kern_all(rows,:) = kb;
        conv_all(rows)   = (out.fODF_regularization.flag_converged(:) == 1);

        fprintf('   Lmax %d, SNR %-4s: %2d coefficients, %6.1f s  (lambda_nonneg = %g, lambda_tikhonov = %g)\n', ...
                Lf, SNR_LABEL{is}, ncoef, el, ...
                out.fODF_regularization.lambda_nonneg, ...
                out.fODF_regularization.lambda_tikhonov);
    end
    SH_SMI{iL} = sh_all; KERN_SMI{iL} = kern_all; CONV_SMI{iL} = conv_all;
end

% The kernel is estimated from rotational invariants, which do not depend on the
% fODF's Lmax in the same way -- so this table should be nearly flat across
% Lmax. Down the SNR axis it is NOT expected to be flat: this is where the noise
% shows up first, well before the fODF does.
knm = {'f','Da','Depar','Deperp','fw'};
fprintf('   kernel recovery, median over the %d voxels of each block (truth in brackets):\n', ...
        NVOX_SNR);
fprintf('        Lmax   SNR  ');
for j = 1:5, fprintf('%16s', sprintf('%s [%.2f]', knm{j}, K(j))); end
fprintf('\n');
for iL = 1:numel(LMAX_LIST)
    for k = 1:NSNR
        is   = SNR_ORD(k);
        rows = find(snr_id == is);
        fprintf('        %4d  %4s  ', LMAX_LIST(iL), SNR_LABEL{is});
        for j = 1:5
            col = KERN_SMI{iL}(rows,j); col = col(isfinite(col));
            fprintf('%16.3f', median(col));
        end
        fprintf('\n');
    end
end
fprintf(['   The kernel is NOT recovered exactly, and the SMI fODF is deconvolved with this\n' ...
         '   ESTIMATED kernel rather than the true one. That is deliberate -- it is what\n' ...
         '   happens on real data. NOTE THE ASYMMETRY: the CSD arms in Step 6b are given the\n' ...
         '   EXACT response, so on this axis the comparison flatters them, not SMI.\n']);

% CHECK. The constrained deconvolution must converge in every voxel at every
% Lmax, and no NaN may reach the fODF: a NaN in an SH volume breaks downstream
% tractography silently.
all_conv = true; all_fin = true;
for iL = 1:numel(LMAX_LIST)
    all_conv = all_conv && all(CONV_SMI{iL} == 1);
    all_fin  = all_fin  && (NONFIN(iL) == 0);
end
fprintf('   CHECK deconvolution converged everywhere               %s\n', VERDICT{1+all_conv});
fprintf('   converged fraction per block:\n        Lmax  ');
for k = 1:NSNR, fprintf('%8s', SNR_LABEL{SNR_ORD(k)}); end
fprintf('\n');
for iL = 1:numel(LMAX_LIST)
    fprintf('        %4d  ', LMAX_LIST(iL));
    for k = 1:NSNR
        rows = find(snr_id == SNR_ORD(k));
        fprintf('%7.1f%%', 100*mean(CONV_SMI{iL}(rows)));
    end
    fprintf('\n');
end
fprintf('   CHECK fODF all finite at every Lmax                    %s\n', VERDICT{1+all_fin});

% CHECK. The p_00 = 1 convention must survive the fit at every Lmax. This is
% also what makes the isotropic subtraction in Step 7 reduce to 1/(4*pi) for
% this arm, which Step 7 checks explicitly.
e_l0 = 0;
for iL = 1:numel(LMAX_LIST)
    e_l0 = max(e_l0, max(abs(SH_SMI{iL}(:,1) - 1/sqrt(4*pi))));
end
fprintf('   CHECK p_00 == 1 convention held   max|err| = %.2e   %s\n\n', ...
        e_l0, VERDICT{1+(e_l0 < 1e-12)});

## Step 6b — the MRtrix arms: `dwi2fod csd` and `dwi2fod msmt_csd`

**This is the part `smi_manuscript_60deg.m` left as a commented stub.** Nothing
below reimplements MRtrix: the same noisy signal Step 5 built is written out as
an MRtrix image with an embedded `dw_scheme`, the binaries are invoked, and
their fODFs are read back.

Four things have to be right, and each gets a check rather than an assumption.

**1. MRtrix must bin the shells the way SMI did.** The protocol's b values
jitter (18 distinct values across 4 shells), and MRtrix does its own shell
clustering. If it split a shell, or ordered them differently, the rows of the
response file would line up with the wrong b values and every arm would be
deconvolved with a response for a different shell. So the shells are read back
out of MRtrix with `mrinfo -shell_bvalues`, checked against SMI's binning, and
**the response is evaluated at MRtrix's own average b for each shell**, not at
the nominal 0/1/2/3.

**2. The response has to be the right object.** `r_l(b) = K_l(b) sqrt((2l+1)4pi)`
is written through `SMI_response_helpers`, read back off disk, and compared
against the array that was written.

**3. Lmax has to match.** `dwi2fod` is given `-lmax` matching `LMAX_LIST`
entry by entry, and one response file is written per Lmax with exactly the right
number of columns, so there is no silent truncation and the ceiling computed in
Step 7 is the correct bound for every arm.

**4. Scale is not comparable, and does not need to be.** An SMI fODF has
`p_00 = 1` and integrates to 1; an MRtrix FOD is unnormalised and its amplitude
carries apparent fibre density. Step 7's peak finder is made scale-free by
subtracting **each voxel's own `l = 0` term** rather than the constant
`1/(4*pi)` — which for the SMI arm is the same number, and is checked to be
bit-identical there.

**MSMT-CSD needs three tissues, and the simulation contains one.** Its second
and third responses are idealized isotropic ones at `D_GM = 0.8` and
`D_FW = 3` µm²/ms. No simulated voxel contains grey matter or free-standing CSF,
so what MSMT assigns to those compartments is a measurement of its own leakage,
and it is printed below rather than discarded.

In [ ]:
% Every arm carries the same three things: a name, one [NVOX x ncoef] SH matrix
% per Lmax, and the Lmax list they were fitted at. Step 7 does not know or care
% which arm produced which matrix.
ARMS = {};
a = struct(); a.name = 'SMI'; a.sh = SH_SMI; ARMS{end+1} = a;

if ~RUN_MRTRIX
    fprintf('Step 6b: SKIPPED (RUN_MRTRIX = false), scoring the SMI arm alone\n\n');
else
if ~exist(MDIR,'dir'), mkdir(MDIR); end
fprintf('Step 6b: the MRtrix arms, in %s\n', MDIR);

% ------------------------------------------------ write the DWI MRtrix reads
% b in s/mm^2, which is what MRtrix expects; this package carries b in ms/um^2
% everywhere else. Voxel order is column-major over GRID_ALL, so voxel v of
% S_noisy is voxel v of the image and Step 7 can index them the same way.
f_dwi  = fullfile(MDIR, [TAG '_dwi']);
f_mask = fullfile(MDIR, [TAG '_mask']);
MR.write(f_dwi,  reshape(S_noisy, [GRID_ALL Ndwi]), ...
         struct('grad', [bvecs bvals(:)*1000]));
MR.write(f_mask, ones(GRID_ALL), struct('datatype','UInt8'));
fprintf('   wrote %s_dwi.mih  [%s x %d] with dw_scheme\n', TAG, mat2str(GRID_ALL), Ndwi);

% CHECK. mrinfo must agree with what we think we wrote. This is the one place
% the locally-implemented image writer is checked against MRtrix itself.
[st, txt] = system(sprintf('mrinfo -size "%s.mih" 2>&1', f_dwi));
sz_mr = sscanf(txt, '%f')';
ok_sz = (st == 0) && numel(sz_mr) == 4 && all(sz_mr == [GRID_ALL Ndwi]);
fprintf('   CHECK mrinfo reads back size %s                 %s\n', ...
        mat2str(sz_mr), VERDICT{1+ok_sz});
if ~ok_sz, fprintf(2, '%s\n', txt); error('mrtrix could not read the image we wrote'); end

% ----------------------------------------- what MRtrix thinks the shells are
[st, txt] = system(sprintf('mrinfo -shell_bvalues "%s.mih" 2>/dev/null', f_dwi));
b_mr = sscanf(txt, '%f')';
[~, txt] = system(sprintf('mrinfo -shell_sizes "%s.mih" 2>/dev/null', f_dwi));
n_mr = sscanf(txt, '%f')';
fprintf('   MRtrix shells: b = %s s/mm^2\n', mat2str(round(b_mr*100)/100));
fprintf('                  n = %s volumes\n', mat2str(n_mr));

% CHECK. Same number of shells, same sizes, same order as SMI's binning, and
% ascending. If this fails, every response row is attached to the wrong shell.
ok_shells = (numel(b_mr) == numel(b_shell)) && all(n_mr == n_shell(:)') && ...
            all(diff(b_mr) > 0);
fprintf('   CHECK MRtrix and SMI agree on the shells (%d, sizes %s)   %s\n', ...
        numel(b_mr), mat2str(n_mr), VERDICT{1+ok_shells});
% The average b of a shell is not the nominal b: the acquired values jitter.
% Report the gap rather than hiding it -- the response is evaluated at MRtrix's
% number, so this is the b the CSD arms actually deconvolve with.
fprintf('        shell   SMI b (ms/um^2)   MRtrix b (ms/um^2)   difference\n');
for i = 1:numel(b_mr)
    fprintf('        %4d    %14.5f   %18.5f   %10.2e\n', ...
            i, b_shell(i), b_mr(i)/1000, b_shell(i) - b_mr(i)/1000);
end

% ------------------------------------------------- the single-shell subset
% dwi2fod csd is single-shell and is run on the top shell, which is where SSST
% CSD is normally run. dwiextract is given MRtrix's own shell b value, not the
% nominal 3000, so it cannot miss a jittered volume.
f_b3 = fullfile(MDIR, [TAG '_b3']);
cmd  = sprintf('dwiextract "%s.mih" -shells 0,%g "%s.mih" -force -quiet 2>&1', ...
               f_dwi, b_mr(end), f_b3);
[st, txt] = system(cmd);
if st ~= 0, fprintf(2,'%s\n',txt); error('dwiextract failed'); end
[~, txt] = system(sprintf('mrinfo -shell_sizes "%s.mih" 2>/dev/null', f_b3));
fprintf('   top-shell subset for SSST-CSD: %s volumes\n', strtrim(txt));

% ----------------------------------------------- responses, one set per Lmax
% The WM response is the EXACT zonal response of the kernel that generated the
% data, evaluated at MRtrix's own shell b values. GM and CSF are idealized
% isotropic responses; no simulated voxel contains either tissue, and they exist
% only because msmt_csd needs at least as many tissues as it has shells to
% separate.
fprintf('   idealized WM response, normalised to l = 0:\n');
r_wm_full = RH.zh(C.K_WM, b_mr/1000, max(LMAX_LIST), C.D_FW);
for i = 1:numel(b_mr)
    fprintf('        b = %7.1f   ', b_mr(i));
    fprintf('%9.4f', r_wm_full(i,:)/r_wm_full(i,1)); fprintf('\n');
end

f_resp = cell(1, numel(LMAX_LIST));
for iL = 1:numel(LMAX_LIST)
    Lf = LMAX_LIST(iL);
    r_wm  = RH.zh(C.K_WM, b_mr/1000, Lf, C.D_FW);          % [nshell x (Lf/2+1)]
    r_gm  = exp(-(b_mr(:)/1000)*D_GM)*sqrt(4*pi);          % [nshell x 1]
    r_csf = exp(-(b_mr(:)/1000)*C.D_FW)*sqrt(4*pi);
    R = struct();
    R.wm    = fullfile(MDIR, sprintf('%s_resp_wm_lmax%d.txt',  TAG, Lf));
    R.gm    = fullfile(MDIR, sprintf('%s_resp_gm.txt',  TAG));
    R.csf   = fullfile(MDIR, sprintf('%s_resp_csf.txt', TAG));
    R.wm_b3 = fullfile(MDIR, sprintf('%s_resp_wm_b3_lmax%d.txt', TAG, Lf));
    RH.write_response(R.wm,    r_wm);
    RH.write_response(R.gm,    r_gm);
    RH.write_response(R.csf,   r_csf);
    RH.write_response(R.wm_b3, r_wm(end,:));               % single shell, one row
    f_resp{iL} = R;

    % CHECK. The file MRtrix will read must hold the numbers we computed. This
    % is a disk round trip, not a comparison of an array with itself.
    e_rt = max(max(abs(RH.read_response(R.wm) - r_wm)));
    fprintf('   CHECK Lmax %d response round trip through disk  max|err| = %.2e   %s\n', ...
            Lf, e_rt, VERDICT{1+(e_rt < 1e-7)});
end

% NOT a check, a measurement. MRtrix deconvolves with ONE response per shell,
% evaluated at the shell's average b, while the data was generated at each
% volume's exact b. That mismatch is a real cost of the jittered protocol and it
% is paid by the CSD arms only -- SMI's kernel fit uses the per-volume b.
r_sh_avg = RH.zh(C.K_WM, b_mr/1000, LMAX_GT, C.D_FW);
r_exact  = RH.zh(C.K_WM, bvals(dw), LMAX_GT, C.D_FW);
sid_dw   = shell_id(dw);
e_jit    = 0;
for i = 1:sum(dw)
    th_i  = linspace(0, pi, 181);
    e_jit = max(e_jit, max(abs(RH.profile(r_exact(i,:), th_i) - ...
                               RH.profile(r_sh_avg(sid_dw(i),:), th_i))));
end
fprintf('   shell-averaging residual: max|R_exact(b_volume) - R(b_shell)| = %.2e\n', e_jit);
fprintf('        (the price the CSD arms pay for b-value jitter; sigma at the\n');
fprintf('         quietest SNR in this sweep is %.4f, so it is %s)\n', ...
        min(SIGMA_LIST(SIGMA_LIST>0)), ...
        VERDICT{1+(e_jit < min(SIGMA_LIST(SIGMA_LIST>0)))});
end

In [ ]:
if RUN_MRTRIX
% --------------------------------------------------------- run the binaries
SH_CSD  = cell(1, numel(LMAX_LIST));
SH_MSMT = cell(1, numel(LMAX_LIST));
tissue_share = nan(numel(LMAX_LIST), 3);

for iL = 1:numel(LMAX_LIST)
    Lf = LMAX_LIST(iL);
    R  = f_resp{iL};
    nc = (Lf/2+1)*(Lf+1);

    f_msmt = fullfile(MDIR, sprintf('%s_msmtfod_lmax%d', TAG, Lf));
    f_mgm  = fullfile(MDIR, sprintf('%s_msmtgm_lmax%d',  TAG, Lf));
    f_mcsf = fullfile(MDIR, sprintf('%s_msmtcsf_lmax%d', TAG, Lf));
    f_csd  = fullfile(MDIR, sprintf('%s_csdfod_lmax%d',  TAG, Lf));

    t0  = tic;
    cmd = sprintf(['dwi2fod msmt_csd "%s.mih" "%s" "%s.mih" "%s" "%s.mih" "%s" "%s.mih" ' ...
                   '-mask "%s.mih" -lmax %d,0,0 -force -quiet 2>&1'], ...
                  f_dwi, R.wm, f_msmt, R.gm, f_mgm, R.csf, f_mcsf, f_mask, Lf);
    [st, txt] = system(cmd);
    if st ~= 0, fprintf(2,'%s\n',txt); error('dwi2fod msmt_csd failed at Lmax %d', Lf); end
    t_msmt = toc(t0);

    t0  = tic;
    cmd = sprintf(['dwi2fod csd "%s.mih" "%s" "%s.mih" -mask "%s.mih" ' ...
                   '-lmax %d -force -quiet 2>&1'], f_b3, R.wm_b3, f_csd, f_mask, Lf);
    [st, txt] = system(cmd);
    if st ~= 0, fprintf(2,'%s\n',txt); error('dwi2fod csd failed at Lmax %d', Lf); end
    t_csd = toc(t0);

    Vm = MR.read([f_msmt '.mih']);  SH_MSMT{iL} = reshape(Vm, [NVOX size(Vm,4)]);
    Vc = MR.read([f_csd  '.mih']);  SH_CSD{iL}  = reshape(Vc, [NVOX size(Vc,4)]);
    Vg = MR.read([f_mgm  '.mih']);  Vf = MR.read([f_mcsf '.mih']);

    fprintf('   Lmax %d: msmt_csd %.1f s, csd %.1f s, %d coefficients each\n', ...
            Lf, t_msmt, t_csd, size(Vm,4));

    % CHECK. The right number of coefficients, and nothing non-finite: a NaN in
    % an SH volume breaks downstream tractography silently.
    ok_nc  = (size(Vm,4) == nc) && (size(Vc,4) == nc);
    ok_fin = all(isfinite(SH_MSMT{iL}(:))) && all(isfinite(SH_CSD{iL}(:)));
    fprintf('   CHECK Lmax %d: %d coefficients as expected, all finite   %s\n', ...
            Lf, nc, VERDICT{1+(ok_nc && ok_fin)});

    % What MSMT put in the two tissues the simulation does not contain. The
    % share is of the l = 0 coefficients, which are on a common scale because
    % all three responses were built from the same S0 = 1 convention. It is a
    % share, not a calibrated volume fraction.
    c0 = [mean(SH_MSMT{iL}(:,1)), mean(Vg(:)), mean(Vf(:))];
    tissue_share(iL,:) = c0/sum(c0);
end

fprintf('   MSMT-CSD tissue share (mean l=0 coefficient, normalised):\n');
fprintf('        Lmax        WM        GM       CSF\n');
for iL = 1:numel(LMAX_LIST)
    fprintf('        %4d  %8.4f  %8.4f  %8.4f\n', LMAX_LIST(iL), tissue_share(iL,:));
end
fprintf(['        Every simulated voxel is pure white matter with fw = %.2f, so anything\n' ...
         '        outside the WM column is leakage. Compare against\n' ...
         '        Reports/REPORT_SMI_deconvolution_MonteCarlo.md section 6.2, which found\n' ...
         '        MSMT''s CSF fraction is NOT a usable free-water estimate.\n'], C.K_WM(5));

a = struct(); a.name = 'SSST-CSD'; a.sh = SH_CSD;  ARMS{end+1} = a;
a = struct(); a.name = 'MSMT-CSD'; a.sh = SH_MSMT; ARMS{end+1} = a;
end

NARM = numel(ARMS);
fprintf('\n   %d arms to score: ', NARM);
for ia = 1:NARM, fprintf('%s  ', ARMS{ia}.name); end
fprintf('\n\n');

## Step 7 — peaks, angular error and spurious peaks, per arm, per Lmax, per SNR

The only questions a tractography algorithm asks of an fODF are "how many
fibres, and pointing where", so those are what get scored — **by the same lines
of code for all three arms**.

Peaks are found by evaluating the fODF on a dense direction set and keeping
every direction not smaller than any neighbour within `PEAK_NBR` degrees, then
keeping those whose *anisotropic* amplitude is at least `PEAK_REL` of the
largest. Subtracting the isotropic part matters: without it a nearly isotropic
fODF looks like it has many strong peaks.

**The isotropic part is each voxel's own `l = 0` term**, `c_00 * Y_00 =
c_00 / sqrt(4 pi)`. For SMI, `c_00 = 1/sqrt(4 pi)` in every voxel, so this is
exactly the constant `1/(4 pi)` the `.m` file subtracts — checked below to be
bit-identical. For an MRtrix FOD, `c_00` varies from voxel to voxel and carries
apparent fibre density, so the constant would be wrong and the general form is
what makes the arms comparable.

Each Lmax is scored against **its own ceiling**: the ground truth truncated to
that same Lmax, with no noise and no fitting. That is what separates "the method
failed" from "this angular order cannot represent the answer". The ceiling is
not scored by a copy of the peak finder — the truth is prepended as the *first
row of every block*, so it goes through byte for byte the same code as the
realisations. It is the same for every arm, which is itself a check.

**Four numbers per cell**, because a sweep is where they stop agreeing:
**correct count** (what tractography consumes), **mean error** (the bias),
**std error** (the spread) and **spurious** (peaks beyond the true count).
"Bias" here is the mean of a non-negative quantity, so it does not go to zero
even for a perfect estimator — the floor is the direction grid and the band
limit. Read it against the `SNR = inf` row of the same curve, not against zero.

In [ ]:
PEAK_NBR = 12;      % degrees, neighbourhood for the local-maximum test
PEAK_REL = 0.30;    % keep peaks at >= 30% of the largest anisotropic amplitude
de   = H.dirs(1500);
ND   = size(de,1);
cosN = cosd(PEAK_NBR);

% The neighbour set of every direction, including itself, precomputed once.
NBIDX = cell(1, ND);
for j = 1:ND
    NBIDX{j} = find((de*de(j,:)') > cosN);    % includes j itself
end

fprintf('Step 7: peaks (%d directions, %d deg neighbourhood, %.0f%% threshold)\n', ...
        ND, PEAK_NBR, 100*PEAK_REL);
fprintf('   The grid spacing sets a floor on angular error of roughly %.1f deg;\n', ...
        sqrt(4*pi/ND)*180/pi/2);
fprintf('   the ceiling line under each condition shows that floor directly.\n');

nL = numel(LMAX_LIST);
res_all  = zeros(NARM, nL, NSNR, NCOND);   % % recovering the true count
bias_all = nan(NARM, nL, NSNR, NCOND);     % mean angular error, deg
sd_all   = nan(NARM, nL, NSNR, NCOND);     % std of angular error, deg
med_all  = nan(NARM, nL, NSNR, NCOND);     % median angular error, deg
spur_all = zeros(NARM, nL, NSNR, NCOND);   % mean spurious peaks per voxel
ceil_n   = zeros(NARM, nL, NCOND);         % peaks the truth itself gives
ceil_err = nan(NARM, nL, NCOND);           % and the truth's own error
P1_all   = cell(NARM, nL);                 % primary peak per voxel, for Step 8

% CHECK. For the SMI arm the general isotropic subtraction must reduce to the
% constant 1/(4*pi) the .m file uses, bit for bit. If it does not, the two files
% are not scoring the same quantity.
Ye_c   = SMI.get_even_SH(de, LMAX_LIST(1), C.CS_PHASE);
SHc    = SH_SMI{1}(1:min(50,NVOX), :);
A_gen  = SHc*Ye_c' - SHc(:,1)*(1/sqrt(4*pi));
A_const= SHc*Ye_c' - 1/(4*pi);
e_iso  = max(abs(A_gen(:) - A_const(:)));
% NOT bit-identical, and it cannot be. SMI's c_00 is 1/sqrt(4*pi), so the
% general form evaluates (1/sqrt(4*pi))*(1/sqrt(4*pi)) where the constant form
% evaluates 1/(4*pi), and those two differ by one rounding step. Measured at
% 5.6e-17, about 4 ulp of 1/(4*pi) = 0.0796 -- twelve orders of magnitude below
% the peak finder's angular resolution. The claim being tested is that the two
% forms agree, not that they are the same expression.
fprintf('   CHECK isotropic subtraction agrees with 1/(4pi) for SMI  max|err| = %.2e   %s\n', ...
        e_iso, VERDICT{1+(e_iso < 1e-15)});

for iL = 1:nL
    Lf = LMAX_LIST(iL);
    Ye = SMI.get_even_SH(de, Lf, C.CS_PHASE);
    nc = size(Ye,2);
    fprintf('\n   Lmax %d\n', Lf);
    fprintf('     %-10s %-9s %5s   correct count   mean err   std err   median err   spurious\n', ...
            'arm', 'condition', 'SNR');

    for ia = 1:NARM
        P1_all{ia,iL} = nan(NVOX, 3);
        for ic = 1:NCOND
            ntrue = numel(axes_gt{ic});
            for k = 1:NSNR
                is   = SNR_ORD(k);
                rows = find(cond_id == ic & snr_id == is);

                % Row 1 is the ground truth truncated to THIS Lmax; rows 2:end
                % are the realisations of this condition at this SNR.
                SHB  = [sh_gt(ic,1:nc); ARMS{ia}.sh{iL}(rows,:)];
                A    = SHB*Ye' - SHB(:,1)*(1/sqrt(4*pi));
                nrow = size(A,1);

                Amax = zeros(nrow, ND);
                for j = 1:ND
                    Amax(:,j) = max(A(:, NBIDX{j}), [], 2);
                end
                ismax = (A > 0) & (A >= Amax);

                nfound = zeros(nrow,1);
                aerr   = nan(nrow,1);
                pk1    = nan(nrow,3);
                for r = 1:nrow
                    lm = find(ismax(r,:)); lm = lm(:);
                    if isempty(lm), continue; end
                    aa = A(r,:)';
                    lm = lm(aa(lm) >= PEAK_REL*max(aa(lm)));
                    [~, o] = sort(aa(lm), 'descend'); lm = lm(o);
                    P = de(lm,:);
                    sel = true(size(P,1),1);
                    for i = 1:size(P,1)
                        if ~sel(i), continue; end
                        dup = abs(P*P(i,:)') > cosN; dup(i) = false; sel(dup) = false;
                    end
                    P = P(sel,:);
                    nfound(r) = size(P,1);
                    pk1(r,:)  = P(1,:);
                    d = zeros(1,ntrue);
                    for kk = 1:ntrue
                        d(kk) = acosd(min(abs(P(1,:)*axes_gt{ic}{kk}'), 1));
                    end
                    aerr(r) = min(d);
                end

                ceil_n(ia,iL,ic)   = nfound(1);          % identical at every SNR
                ceil_err(ia,iL,ic) = aerr(1);
                P1_all{ia,iL}(rows,:) = pk1(2:end,:);
                nf  = nfound(2:end);
                ae  = aerr(2:end);
                fin = isfinite(ae);

                res_all(ia,iL,is,ic)  = 100*mean(nf == ntrue);
                spur_all(ia,iL,is,ic) = mean(max(nf - ntrue, 0));
                if any(fin)
                    bias_all(ia,iL,is,ic) = mean(ae(fin));
                    med_all(ia,iL,is,ic)  = median(ae(fin));
                end
                if sum(fin) > 1
                    sd_all(ia,iL,is,ic) = std(ae(fin));
                end

                fprintf('     %-10s %-9s %5s   %11.1f%%   %8.2f  %8.2f     %8.2f   %8.3f\n', ...
                        ARMS{ia}.name, COLHDR{ic}, SNR_LABEL{is}, ...
                        res_all(ia,iL,is,ic), bias_all(ia,iL,is,ic), ...
                        sd_all(ia,iL,is,ic), med_all(ia,iL,is,ic), ...
                        spur_all(ia,iL,is,ic));
            end
        end
    end

    % The ceiling is a property of the truth and the band limit, so every arm
    % must report the same one. It is prepended to each arm's block separately,
    % so agreement here is a real check that the blocks are aligned.
    for ic = 1:NCOND
        ntrue = numel(axes_gt{ic});
        if ceil_n(1,iL,ic) == ntrue, ctxt = 'resolvable';
        else, ctxt = sprintf('%d of %d -- NOT resolvable', ceil_n(1,iL,ic), ntrue); end
        fprintf('     %-10s %-9s truth   %12s   %8.2f                          %8.3f   <- ceiling, %s\n', ...
                'all arms', COLHDR{ic}, '--', ceil_err(1,iL,ic), ...
                max(ceil_n(1,iL,ic)-ntrue, 0), ctxt);
    end
    ok_ceil = all(all(ceil_n(:,iL,:) == repmat(ceil_n(1,iL,:), NARM, 1, 1))) && ...
              max(max(abs(ceil_err(:,iL,:) - repmat(ceil_err(1,iL,:), NARM, 1, 1)))) < 1e-9;
    fprintf('     CHECK every arm reports the same ceiling                      %s\n', ...
            VERDICT{1+ok_ceil});
end

fprintf('\n');
fprintf('   Reading this table: a low "correct count" next to a ceiling that says NOT\n');
fprintf('   resolvable is the angular order failing, not the method. A low count next to\n');
fprintf('   a resolvable ceiling is the method or the noise, and the SNR column says\n');
fprintf('   which -- if it is still low at SNR inf, noise was never the problem.\n\n');

In [ ]:
% CHECK. The noise-free arm must recover the true fibre count wherever the truth
% itself resolves. With no noise the only things left between the fit and the
% truth are the band limit and the response, so a failure here makes nothing at
% a finite SNR interpretable. Conditions whose ceiling is already below the true
% count are SKIPPED, not failed.
%
% Also: at sigma = 0 every realisation of a condition is the same signal vector,
% and every method here is deterministic, so the whole block must come back bit
% identical and its standard deviation must be exactly 0.
i_inf = find(isinf(SNR_LIST), 1);
if ~isempty(i_inf)
    det_max = zeros(1, NARM);
    for ia = 1:NARM
        ok_nf = true; n_tested = 0; ok_det = true;
        for iL = 1:nL
            for ic = 1:NCOND
                if ceil_n(ia,iL,ic) == numel(axes_gt{ic})
                    ok_nf = ok_nf && (res_all(ia,iL,i_inf,ic) > 99);
                    n_tested = n_tested + 1;
                end
                % Determinism is tested on the COEFFICIENTS, not on
                % std(angular error). std() computes sum(x)/n first, and for n
                % identical values that division need not return the value
                % exactly, so a deterministic block can produce std ~1e-16 and
                % fail a == 0 test for reasons that have nothing to do with
                % determinism.
                %
                % The tolerance is NOT bit-equality, and that is a measured
                % result rather than a hedge. At sigma = 0 every voxel of a
                % block is fed a bit-identical signal, and the non-negativity
                % constraint converges in the same number of iterations in
                % every one of them -- yet SMI returns THREE distinct answers
                % across 27 voxels on a [3 3 3] grid, i.e. one per slice,
                % differing by 1.6e-12 in plm and 1.1e-11 in the kernel. That
                % is BLAS blocking: the reduction order inside the matrix
                % operations depends on where a voxel sits in the array, and
                % the last few ulps follow. It is twelve orders of magnitude
                % below the peak finder's 2.6 degree grid resolution and
                % changes no result here.
                %
                % Both MRtrix arms ARE bit identical, because dwi2fod works one
                % voxel at a time. So this line separates two real things: a
                % method that is exactly reproducible voxel to voxel, and one
                % that is reproducible only to floating point.
                Bdet  = ARMS{ia}.sh{iL}(snr_id == i_inf & cond_id == ic, :);
                e_det = max(max(abs(Bdet - repmat(Bdet(1,:), size(Bdet,1), 1))));
                det_max(ia) = max(det_max(ia), e_det);
                ok_det = ok_det && (e_det < 1e-9);
            end
        end
        fprintf('   CHECK %-9s noise-free arm recovers the true count (%d of %d cells resolve)  %s\n', ...
                ARMS{ia}.name, n_tested, nL*NCOND, VERDICT{1+ok_nf});
        if det_max(ia) == 0, dtxt = 'bit identical';
        else, dtxt = sprintf('max|err| = %.1e', det_max(ia)); end
        fprintf('   CHECK %-9s noise-free block identical row to row, %-22s %s\n', ...
                ARMS{ia}.name, dtxt, VERDICT{1+ok_det});
    end
end

% CHECK. More noise must not help. Compared at the extremes rather than pairwise
% so that Monte Carlo error between adjacent SNRs cannot trip it.
is_lo = SNR_ORD(1); is_hi = SNR_ORD(end);
for ia = 1:NARM
    ok_mono = true;
    for iL = 1:nL
        for ic = 1:NCOND
            ok_mono = ok_mono && (bias_all(ia,iL,is_hi,ic) <= bias_all(ia,iL,is_lo,ic));
        end
    end
    fprintf('   CHECK %-9s mean angular error at SNR %s <= that at SNR %s          %s\n', ...
            ARMS{ia}.name, SNR_LABEL{is_hi}, SNR_LABEL{is_lo}, VERDICT{1+ok_mono});
end
fprintf('\n');

## Step 8 — export, and an independent peak finder

Everything above is scored by this notebook. That is fine for reading, but it is
not independent: the same code wrote the SMI fODF and found its peaks. This step
hands every arm's fODF to **`sh2peaks`**, which has never seen this package, and
compares its primary peak against the one found above.

The fit ran at `CS_phase = 0`, where SMI's spherical harmonic basis *is*
MRtrix's, so the coefficients need no conversion — they are written straight out
as MRtrix SH images. That is the claim `check_mrtrix_basis.sh` establishes in
the abstract; this is it applied to the actual data.

`export/voxel_key_<preset>.txt` records which voxel was which condition and
which noise level, so a peak found in MRtrix can be matched back.

In [ ]:
edir = fullfile(pkgdir, 'export');
if ~exist(edir, 'dir'), mkdir(edir); end
fprintf('Step 8: writing MRtrix SH images to %s\n', edir);

for iL = 1:nL
    Lf = LMAX_LIST(iL);
    fn = fullfile(edir, sprintf('smifod_%s_lmax%d', C.PRESET, Lf));
    MR.write(fn, reshape(SH_SMI{iL}, [GRID_ALL size(SH_SMI{iL},2)]));
    fprintf('   smifod_%s_lmax%d.mih  [%s x %d]\n', C.PRESET, Lf, ...
            mat2str(GRID_ALL), size(SH_SMI{iL},2));
end

% A voxel-order key. Column 1 is the linear voxel index in MRtrix's order,
% column 2 the SNR, column 3 the crossing angle, columns 4-6 and 7-9 the true
% fibre axes.
key = zeros(NVOX, 9);
for v = 1:NVOX
    ic = cond_id(v);
    key(v,1) = v;
    key(v,2) = SNR_LIST(snr_id(v));
    key(v,3) = C.ANGLES(ic);
    key(v,4:6) = axes_gt{ic}{1};
    if numel(axes_gt{ic}) == 2, key(v,7:9) = axes_gt{ic}{2}; end
end
fid = fopen(fullfile(edir, sprintf('voxel_key_%s.txt', C.PRESET)), 'w');
fprintf(fid, '%% voxel  SNR  crossing_deg  axis1_x axis1_y axis1_z  axis2_x axis2_y axis2_z\n');
fprintf(fid, '%% axis2 is 0 0 0 for the single fibre condition. SNR Inf is the noise-free arm.\n');
fprintf(fid, '%% Voxel order is column-major over the %s grid, matching the .mih images:\n', ...
        mat2str(GRID_ALL));
fprintf(fid, '%% %d contiguous blocks of %d voxels, one per SNR, in the order %s.\n', ...
        NSNR, NVOX_SNR, strjoin(SNR_LABEL, ', '));
fprintf(fid, '%d %g %d %.9f %.9f %.9f %.9f %.9f %.9f\n', key');
fclose(fid);
fprintf('   voxel_key_%s.txt  SNR and true fibre axes per voxel\n', C.PRESET);

In [ ]:
% --------------------------- the independent check: sh2peaks on every arm
if RUN_SH2PEAKS
    fprintf('\n   sh2peaks against this notebook''s peak finder:\n');
    fprintf('        %-10s %5s %5s   mean |delta| primary peak   worst   agree < 5 deg\n', ...
            'arm', 'Lmax', 'SNR');
    for iL = 1:nL
        Lf = LMAX_LIST(iL);
        for ia = 1:NARM
            src = fullfile(MDIR, sprintf('%s_sh2p_src_%s_lmax%d', TAG, ...
                                         strrep(ARMS{ia}.name,'-','_'), Lf));
            pkf = fullfile(MDIR, sprintf('%s_sh2p_pk_%s_lmax%d', TAG, ...
                                         strrep(ARMS{ia}.name,'-','_'), Lf));
            MR.write(src, reshape(ARMS{ia}.sh{iL}, [GRID_ALL size(ARMS{ia}.sh{iL},2)]));
            cmd = sprintf('sh2peaks "%s.mih" "%s.mih" -num 4 -force -quiet 2>&1', src, pkf);
            [st, txt] = system(cmd);
            if st ~= 0, fprintf(2,'%s\n',txt); error('sh2peaks failed'); end
            Vp = MR.read([pkf '.mih']);
            Pk = reshape(Vp, [NVOX size(Vp,4)]);
            p1 = Pk(:,1:3);
            n1 = sqrt(sum(p1.^2, 2));
            % Reported PER SNR. Aggregated over the sweep this number is
            % meaningless: where an arm is producing spurious peaks of similar
            % amplitude, "the primary peak" is genuinely ambiguous and two peak
            % finders will disagree by tens of degrees without either being
            % wrong. Splitting by SNR shows that immediately.
            for k = 1:NSNR
                is  = SNR_ORD(k);
                sel = (snr_id == is) & isfinite(n1) & (n1 > 0) & ...
                      all(isfinite(P1_all{ia,iL}), 2);
                if ~any(sel), continue; end
                u = p1(sel,:) ./ repmat(n1(sel), 1, 3);
                d = acosd(min(abs(sum(u .* P1_all{ia,iL}(sel,:), 2)), 1));
                fprintf('        %-10s %5d %5s   %16.2f  %6.2f   %5.1f%% of %d\n', ...
                        ARMS{ia}.name, Lf, SNR_LABEL{is}, mean(d), max(d), ...
                        100*mean(d < 5), sum(sel));
            end
        end
    end
    fprintf(['        sh2peaks maximises over its own grid with its own threshold, so exact\n' ...
             '        agreement is not expected. What IS being tested is that both find the\n' ...
             '        same lobe: a basis or convention error would show up as tens of degrees\n' ...
             '        at EVERY SNR, not just where the fODF is a mess. Read this column\n' ...
             '        against the spurious-peak column of Step 7 -- an arm inventing peaks\n' ...
             '        has no well defined primary peak for either finder to agree on.\n']);
end

fprintf('\n   To look at these outside this notebook, from %s:\n', edir);
fprintf('     sh2peaks smifod_%s_lmax%d.mih peaks.mih -num 4\n', C.PRESET, LMAX_LIST(1));
fprintf('     mrview   smifod_%s_lmax%d.mih -odf.load_sh smifod_%s_lmax%d.mih\n\n', ...
        C.PRESET, LMAX_LIST(1), C.PRESET, LMAX_LIST(1));

## Figures

Four figures. Set `MAKE_FIGURES = false` to skip them.

All glyphs use the renderer MRtrix's `shview` logic implies: **radius is
|amplitude|, colour is the signed amplitude**, so negative lobes show as a
colour change instead of being folded silently into the surface. The
band-limited truth really is negative over much of the sphere (Step 2), and a
renderer that hid that would hide the most surprising result in the notebook.

| figure | layout |
|---|---|
| 1 | the ground truth fODF, then the kernel's response glyph per shell. **One shared radial scale**, so size carries meaning |
| 2 | the signal as surfaces on the sphere: b down the rows, SNR across |
| 3 | reconstructed fODFs, one figure per Lmax: **one row per arm**, truth in column 1 then one column per SNR |
| 4 | bias, spread and spurious peak count against SNR — one curve per Lmax, **one row per arm**, y limits shared per column |

**Only Figure 1 uses a shared radial scale.** Getting that to work takes two
things and only one is obvious: `axis equal` fixes a panel's aspect *ratio*, not
its *limits*, so a glyph drawn at half the radius gets an axis range half as
wide and lands on the page at exactly the same size. Figure 1 scales every glyph
**and** pins each panel to `GLYPH_LIM`. Everywhere else each panel autoscales
deliberately — Figures 2 and 3 are about *shape*, and shrinking one into
illegibility would buy nothing.

Each figure is wrapped in `try`/`catch` and prints `** FIGURE FAILED **` with
the error if it cannot draw. That is not defensive padding: this repository's
Octave container has only the `gnuplot` toolkit, which ignores `camlight` and
`lighting`, and the numeric tables above are the deliverable either way. A
failure here is visible, never silent.

In [ ]:
MAKE_FIGURES = true;
ISO_VIEW     = [1 1 1];      % isometric camera: equal foreshortening on x, y, z
GLYPH_LIM    = [-1.02 1.02]; % axis limits, applied to FIGURE 1 ONLY

if MAKE_FIGURES
    [THg, PHg, dirs_g] = RH.grid(GLYPH_N, GLYPH_N);
    Yg  = SMI.get_even_SH(dirs_g, LMAX_GT, C.CS_PHASE);
    nsh = numel(b_shell);
    ic  = 1;                                   % the only condition: 60 degrees
    fprintf('Figures: glyph mesh %dx%d, toolkit %s\n', GLYPH_N, GLYPH_N, graphics_toolkit());
end

In [ ]:
% ================================================================
% FIGURE 1 -- the ground truth fODF and the kernel's response per shell
% ================================================================
% The left panel is the fODF being convolved; to its right, one glyph per shell
% of what it is convolved with. EVERY RESPONSE GLYPH SHARES ONE RADIAL SCALE, so
% size carries meaning: a smaller glyph is a genuinely smaller signal. The
% b = 0 glyph is a unit sphere for any kernel, because S(0)/S0 = 1 exactly, and
% it is what sets the scale.
%
% The ground truth is deliberately NOT on the response scale: an fODF and a
% signal are different quantities and a shared scale between them would mean
% nothing. It is normalised to its own peak instead.
%
% This is the panel built to take an ESTIMATED response: dwi2response writes
% zonal coefficients in exactly the form RH.zh_glyph draws, so a response
% estimated from data becomes another row here with no conversion.
if MAKE_FIGURES
try
    th_prof = linspace(0, pi, 361);
    r_sh = cell(1, nsh); rmax = 0;
    for i = 1:nsh
        r_sh{i} = RH.zh(C.K_WM, b_shell(i), LMAX_GT, C.D_FW);
        rmax = max(rmax, max(abs(RH.profile(r_sh{i}, th_prof))));
    end
    if rmax <= 0, rmax = 1; end

    figure('Name','Fig 1  ground truth fODF and the kernel response per shell');
    subplot(1, 1+nsh, 1);
    gt_pk = max(abs(sh_gt(ic,:) * Yg')); if gt_pk <= 0, gt_pk = 1; end
    [X,Y,Z,Cc] = RH.sh_glyph(plm_gt(ic,:), LMAX_GT, C.CS_PHASE, GLYPH_N, GLYPH_N, 1/gt_pk);
    surf(X,Y,Z,Cc); shading interp; axis equal off vis3d; view(ISO_VIEW);
    set(gca,'XLim',GLYPH_LIM,'YLim',GLYPH_LIM,'ZLim',GLYPH_LIM);
    camlight headlight; lighting gouraud;
    title(sprintf('truth, %g deg', C.ANGLES(ic)));

    for i = 1:nsh
        subplot(1, 1+nsh, 1+i);
        [X,Y,Z,Cc] = RH.zh_glyph(r_sh{i}, GLYPH_N, GLYPH_N, 1/rmax);
        surf(X,Y,Z,Cc); shading interp; axis equal off vis3d; view(ISO_VIEW);
        set(gca,'XLim',GLYPH_LIM,'YLim',GLYPH_LIM,'ZLim',GLYPH_LIM);
        camlight headlight; lighting gouraud;
        title(sprintf('b = %.0f', b_shell(i)));
    end

    fprintf('   Fig 1: response peak amplitude on the shared scale (max = %.3f)\n', rmax);
    fprintf('        b       peak\n');
    for i = 1:nsh
        fprintf('     %4.2f   %8.4f\n', b_shell(i), ...
                max(abs(RH.profile(r_sh{i}, th_prof))));
    end
catch err
    fprintf(2, '   ** FIGURE 1 FAILED ** %s\n', err.message);
end
end

In [ ]:
% ================================================================
% FIGURE 2 -- the spherical signal: b down the rows, SNR across
% ================================================================
% Radius is S(u)/S0. The noise-free column is the model evaluated on a dense
% grid; every other column is the spherical harmonic fit of ONE representative
% realisation at that SNR, which is the same projection SMI does internally, so
% what is drawn is what the fit actually sees. One shared radial scale.
if MAKE_FIGURES
try
    sig = cell(nsh-1, NSNR); smax = 0;
    for j = 2:nsh
        for kk = 1:NSNR
            is = SNR_ORD(kk);
            if isinf(SNR_LIST(is))
                bq = b_shell(j)*ones(1, size(dirs_g,1));
                sg = H.signal(plm_gt(ic,:), [C.K_WM 1 1], bq, ones(size(bq)), ...
                              zeros(size(bq)), dirs_g, LMAX_GT, C.CS_PHASE, C.D_FW);
                A = reshape(sg, size(THg));
            else
                v  = find(snr_id(:) == is, 1);
                m  = (shell_id(:)' == j);
                Ym = SMI.get_even_SH(bvecs(m,:), LMAX_GT, C.CS_PHASE);
                cm = Ym \ S_noisy(v, m)';
                A  = reshape(Yg*cm, size(THg));
            end
            sig{j-1, kk} = A;
            smax = max(smax, max(abs(A(:))));
        end
    end
    figure('Name','Fig 2  the spherical signal');
    for j = 2:nsh
        for kk = 1:NSNR
            subplot(nsh-1, NSNR, (j-2)*NSNR + kk);
            [X,Y,Z,Cc] = RH.glyph(sig{j-1,kk}, THg, PHg, 1/smax);
            surf(X,Y,Z,Cc); shading interp; axis equal off vis3d; view(ISO_VIEW);
            camlight headlight; lighting gouraud;
            title(sprintf('b %.0f, SNR %s', b_shell(j), SNR_LABEL{SNR_ORD(kk)}));
        end
    end
    fprintf('   Fig 2 drawn (%d shells x %d SNR)\n', nsh-1, NSNR);
catch err
    fprintf(2, '   ** FIGURE 2 FAILED ** %s\n', err.message);
end
end

In [ ]:
% ================================================================
% FIGURE 3 -- reconstructed fODFs, one figure per Lmax, ONE ROW PER ARM
% ================================================================
% Column 1 is the band-limited ground truth at that Lmax -- the bound nothing to
% its right can beat -- then one column per SNR, worst first. Every panel
% autoscales to its own peak, because an SMI fODF and an MRtrix FOD live on
% different scales by construction and the comparison here is of SHAPE.
if MAKE_FIGURES
try
    for iL = 1:nL
        Lf = LMAX_LIST(iL);
        nc = (Lf/2+1)*(Lf+1);
        figure('Name', sprintf('Fig 3  reconstructed fODFs, Lmax %d', Lf));
        for ia = 1:NARM
            subplot(NARM, NSNR+1, (ia-1)*(NSNR+1) + 1);
            amp = reshape(sh_gt(ic,1:nc) * Yg(:,1:nc)', size(THg));
            [X,Y,Z,Cc] = RH.glyph(amp, THg, PHg, 1/max(abs(amp(:))));
            surf(X,Y,Z,Cc); shading interp; axis equal off vis3d; view(ISO_VIEW);
            camlight headlight; lighting gouraud;
            title(sprintf('%s: truth', ARMS{ia}.name));

            for kk = 1:NSNR
                is = SNR_ORD(kk);
                subplot(NARM, NSNR+1, (ia-1)*(NSNR+1) + 1 + kk);
                v   = find(snr_id(:) == is, 1);
                amp = reshape(ARMS{ia}.sh{iL}(v,:) * Yg(:,1:nc)', size(THg));
                pk  = max(abs(amp(:))); if pk <= 0, pk = 1; end
                [X,Y,Z,Cc] = RH.glyph(amp, THg, PHg, 1/pk);
                surf(X,Y,Z,Cc); shading interp; axis equal off vis3d; view(ISO_VIEW);
                camlight headlight; lighting gouraud;
                title(sprintf('SNR %s', SNR_LABEL{is}));
            end
        end
    end
    fprintf('   Fig 3 drawn (%d Lmax x %d arms x %d columns)\n', nL, NARM, NSNR+1);
catch err
    fprintf(2, '   ** FIGURE 3 FAILED ** %s\n', err.message);
end
end

In [ ]:
% ================================================================
% FIGURE 4 -- bias, spread and spurious peaks against SNR
% ================================================================
% ONE ROW PER ARM, three columns: mean angular error, its standard deviation,
% and the mean spurious peak count. One curve per Lmax. All rows share y limits
% per column, so the arms are read off the same axis rather than each being
% autoscaled to look similar. Drawn from exactly the arrays Step 7 printed, so
% the tables and the plots cannot disagree.
if MAKE_FIGURES
try
    figure('Name','Fig 4  bias, spread and spurious peaks against SNR');
    xs   = 1:NSNR;
    mets = {'bias','sd','spur'};
    Ms   = {bias_all, sd_all, spur_all};
    labs = {'mean angular error (deg)', 'std of angular error (deg)', ...
            'spurious peaks per voxel'};
    ylim_all = cell(1,3);
    for im = 1:3
        hi = max(max(max(Ms{im}(:,:,SNR_ORD,ic))));
        if ~isfinite(hi) || hi <= 0, hi = 1; end
        ylim_all{im} = [0 1.05*hi];
    end
    for ia = 1:NARM
        for im = 1:3
            subplot(NARM, 3, (ia-1)*3 + im);
            hold on;
            for iL = 1:nL
                plot(xs, squeeze(Ms{im}(ia, iL, SNR_ORD, ic)), '-o', 'LineWidth', 1.5);
            end
            set(gca, 'XTick', xs, 'XTickLabel', SNR_LABEL(SNR_ORD));
            xlim([0.5 NSNR+0.5]); ylim(ylim_all{im});
            xlabel('SNR'); ylabel(labs{im}); grid on;
            title(ARMS{ia}.name);   % the metric is on the ylabel; a title
                                    % carrying both collides with its neighbour
            if ia == 1 && im == 1
                legend(arrayfun(@(L) sprintf('Lmax %d', L), LMAX_LIST, ...
                                'UniformOutput', false), 'Location', 'best');
            end
        end
    end
    fprintf('   Fig 4 drawn (%d arms x 3 metrics)\n', NARM);
catch err
    fprintf(2, '   ** FIGURE 4 FAILED ** %s\n', err.message);
end
end

In [ ]:
% ======================= summary tables, all arms =======================
fprintf('\n=== simulation complete ===\n');
fprintf('%d realisations per condition per SNR, %d SNR levels, Lmax %s, %d arms\n\n', ...
        NREP, NSNR, mat2str(LMAX_LIST), NARM);
if SMOKE_TEST
    fprintf('*** SMOKE_TEST = true. These numbers are indicative, not publishable. ***\n');
    fprintf('*** Set SMOKE_TEST = false for the manuscript configuration.          ***\n\n');
end

sumnames = {'correct fibre count (%)', 'bias: mean angular error (deg)', ...
            'std of angular error (deg)', 'spurious peaks per voxel'};
sums = {res_all, bias_all, sd_all, spur_all};
for im = 1:numel(sums)
    fprintf('%s\n', sumnames{im});
    for ic = 1:NCOND
        fprintf('  %s\n', COLHDR{ic});
        for ia = 1:NARM
            fprintf('    %-10s   SNR ', ARMS{ia}.name);
            for k = 1:NSNR, fprintf('%10s', SNR_LABEL{SNR_ORD(k)}); end
            fprintf('\n');
            for iL = 1:nL
                fprintf('               Lmax %d ', LMAX_LIST(iL));
                for k = 1:NSNR
                    fprintf('%10.3f', sums{im}(ia, iL, SNR_ORD(k), ic));
                end
                fprintf('\n');
            end
        end
    end
    fprintf('\n');
end

## Step 9 — from here to a full campaign

**Set `SMOKE_TEST = false`.** That is the whole difference between this notebook
and the manuscript configuration: `NREP = 1000`, seven SNR levels and three
angular orders, which is 21 `SMI.fit` calls and runs in hours. The two MRtrix
arms scale with voxel count rather than with fit count and stay in the seconds.

**What to change first if you want to probe it.** Everything is in the
Configuration cell and nothing outside it needs editing: `NREP` sets the
runtime, `SNR_LIST` and `LMAX_LIST` the sweep, `K_WM` the tissue, `KAPPA` the
fibre dispersion, `ANGLES` the crossing, `REG` the regularizer, `PROTOCOL` the
acquisition. Those settings are **local to this notebook** — changing them does
not change `gen_montecarlo.m` or `sweep_nonneg.m`, which keep their own values
in `mc_config.m`.

**Swapping in an estimated response.** Step 6b builds the WM response from the
kernel analytically. To use a response estimated from data instead, replace the
`RH.zh(...)` call with a `RH.read_response(...)` of a `dwi2response` output —
the arrays have the same shape, one row per shell and one column per even l. The
rest of the notebook does not change. Two things to know before doing it:

- **An estimated response is 15–40% blunter than the exact one** at every l
  (`Reports/REPORT_SMI_deconvolution_MonteCarlo.md`, section 6.3), because it
  absorbs fibre dispersion. Three different estimators agree with each other far
  more closely than any of them agrees with the kernel.
- **The committed responses in `deconv_comparison/mrtrix_responses/` were
  estimated on the superseded synthetic protocol** (`# Shells: 0,1000,2000,3000`,
  no jitter), not the real HCP scheme this notebook uses. `gen_phantom.m` still
  reads its acquisition from `binio`'s untracked `data/`, so it must be moved
  onto `protocol/hcp_real_3shell.txt` before its responses can be used here.

**What is deliberately not here.** No edema kernel — this is the healthy
baseline. No estimated response — the arms are given the exact one so that a
difference between them is the deconvolution and nothing else. No real data:
every number in this notebook and in every report in this repository is
simulation.